# Tutorial: Colab Agent

Audience:
- Anyone who wants Google Colab to watch a Google Drive folder for Python script and notebook jobs.

Prerequisites:
- A Google account with access to Colab and Google Drive.
- A Drive folder named `Colab Agent` in `MyDrive`.

Learning goals:
- Mount Google Drive in Colab.
- Create a simple job queue backed by Drive folders.
- Execute single-file jobs or bundled folder jobs dropped into `jobs/inbox/` and save logs back to Drive.


## Workflow

1. Run the setup cells once per Colab session.
2. Drop Python scripts, notebooks, or job folders into `MyDrive/Colab Agent/jobs/inbox/`.
3. For a multi-file job folder, add `job.json` with an `entrypoint` field if there is more than one runnable file; the entrypoint may be either a `.py` file or an `.ipynb` file.
4. Start the watcher cell and leave the runtime connected.
5. Read results in `jobs/done/` and `jobs/failed/`; each job folder contains both the result JSON and the execution log.

Expected folder layout:

```text
Colab Agent/
  jobs/
    inbox/
    running/
    done/
    failed/
```


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('This notebook is intended to run inside Google Colab.') from exc

drive.mount('/content/drive')

if importlib.util.find_spec('papermill') is None:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'papermill'],
        check=True,
    )
    print('Installed papermill for notebook execution.')
else:
    print('papermill is already available.')

ROOT = Path('/content/drive/MyDrive/Colab Agent')
JOBS_ROOT = ROOT / 'jobs'
INBOX_DIR = JOBS_ROOT / 'inbox'
RUNNING_DIR = JOBS_ROOT / 'running'
DONE_DIR = JOBS_ROOT / 'done'
FAILED_DIR = JOBS_ROOT / 'failed'
CONTROL_PATH = ROOT / 'control.md'

for folder in [ROOT, JOBS_ROOT, INBOX_DIR, RUNNING_DIR, DONE_DIR, FAILED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if not CONTROL_PATH.exists():
    CONTROL_PATH.write_text('run\n', encoding='utf-8')

print(f'Watching folder: {INBOX_DIR}')


In [ ]:
POLL_SECONDS = 15
HEARTBEAT_INTERVAL_SECONDS = 30
NOTEBOOK_PROGRESS_INTERVAL_SECONDS = 15
JOB_TIMEOUT_SECONDS = 60 * 30
HEARTBEAT_PATH = ROOT / 'heartbeat.json'
CONTROL_STOP_TOKEN = 'stop'


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def timestamp() -> str:
    return utc_now().strftime('%Y%m%dT%H%M%SZ')


SUPPORTED_SUFFIXES = {'.py', '.ipynb'}


def list_pending_jobs() -> list[Path]:
    return sorted(
        path for path in INBOX_DIR.iterdir()
        if path.is_dir() or (path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES)
    )


def load_job_spec(job_root: Path) -> dict:
    spec_path = job_root / 'job.json'
    if not spec_path.exists():
        return {}
    data = json.loads(spec_path.read_text(encoding='utf-8'))
    if not isinstance(data, dict):
        raise ValueError(f'job.json must contain a JSON object: {spec_path}')
    return data


def resolve_job_entrypoint(job_root: Path) -> Path:
    if job_root.is_file():
        return job_root
    spec = load_job_spec(job_root)
    entrypoint = spec.get('entrypoint')
    if entrypoint:
        candidate = (job_root / str(entrypoint)).resolve()
        if job_root.resolve() not in candidate.parents and candidate != job_root.resolve():
            raise ValueError(f'Entrypoint must stay inside the job folder: {entrypoint}')
        if not candidate.exists() or not candidate.is_file():
            raise FileNotFoundError(f'Entrypoint not found: {entrypoint}')
        if candidate.suffix.lower() not in SUPPORTED_SUFFIXES:
            raise ValueError(f'Unsupported entrypoint type: {candidate.suffix}')
        return candidate
    candidates = sorted(
        path for path in job_root.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES
    )
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise ValueError(f'No supported entrypoint found in job folder: {job_root.name}')
    raise ValueError(
        f"Multiple runnable files found in {job_root.name}; add job.json with an 'entrypoint'."
    )


def build_job_paths(job_path: Path) -> tuple[str, Path, Path]:
    job_id = f"{timestamp()}_{job_path.stem}"
    job_dir = RUNNING_DIR / job_id
    work_root = job_dir / job_path.name
    return job_id, job_dir, work_root


def build_job_command(entrypoint: Path, job_dir: Path) -> list[str]:
    suffix = entrypoint.suffix.lower()
    if suffix == '.py':
        return [sys.executable, str(entrypoint)]
    if suffix == '.ipynb':
        executed_name = f"{entrypoint.stem}.executed{entrypoint.suffix}"
        return [
            sys.executable,
            '-m',
            'papermill',
            str(entrypoint.relative_to(job_dir)),
            str(entrypoint.with_name(executed_name).relative_to(job_dir)),
        ]
    raise ValueError(f'Unsupported job type: {entrypoint.suffix}')


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2) + '\n', encoding='utf-8')


def read_control_text() -> str:
    return CONTROL_PATH.read_text(encoding='utf-8') if CONTROL_PATH.exists() else ''


def stop_requested() -> bool:
    return CONTROL_STOP_TOKEN in read_control_text().lower()


def get_memory_stats() -> dict | None:
    meminfo_path = Path('/proc/meminfo')
    if not meminfo_path.exists():
        return None
    values = {}
    for line in meminfo_path.read_text(encoding='utf-8').splitlines():
        if ':' not in line:
            continue
        key, raw_value = line.split(':', 1)
        parts = raw_value.strip().split()
        if not parts:
            continue
        try:
            values[key] = int(parts[0]) * 1024
        except ValueError:
            continue
    total = values.get('MemTotal')
    available = values.get('MemAvailable')
    if total is None or available is None:
        return None
    used = total - available
    return {
        'total_bytes': total,
        'available_bytes': available,
        'used_bytes': used,
        'usage_percent': round((used / total) * 100, 2) if total else None,
    }


def get_gpu_stats() -> dict | None:
    try:
        result = subprocess.run(
            [
                'nvidia-smi',
                '--query-gpu=name,memory.total,memory.used,utilization.gpu,temperature.gpu',
                '--format=csv,noheader,nounits',
            ],
            capture_output=True,
            text=True,
            check=True,
        )
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None
    gpus = []
    for line in result.stdout.splitlines():
        parts = [part.strip() for part in line.split(',')]
        if len(parts) != 5:
            continue
        name, mem_total, mem_used, utilization, temperature = parts
        gpus.append({
            'name': name,
            'memory_total_mb': int(mem_total),
            'memory_used_mb': int(mem_used),
            'utilization_gpu_percent': int(utilization),
            'temperature_c': int(temperature),
        })
    return {'count': len(gpus), 'devices': gpus}


def get_runtime_stats() -> dict:
    disk = shutil.disk_usage(ROOT)
    stats = {
        'python_executable': sys.executable,
        'cwd': os.getcwd(),
        'disk_total_bytes': disk.total,
        'disk_used_bytes': disk.used,
        'disk_free_bytes': disk.free,
        'memory': get_memory_stats(),
        'gpu': get_gpu_stats(),
    }
    try:
        load1, load5, load15 = os.getloadavg()
        stats['load_average'] = {'1m': load1, '5m': load5, '15m': load15}
    except (AttributeError, OSError):
        stats['load_average'] = None
    return stats


def format_bytes(num_bytes: int | None) -> str:
    if num_bytes is None:
        return 'n/a'
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    value = float(num_bytes)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f'{value:.1f} {unit}'
        value /= 1024
    return f'{value:.1f} TB'


def format_runtime_brief(runtime: dict | None) -> str:
    if not runtime:
        return 'RAM n/a | GPU n/a'
    memory = runtime.get('memory') or {}
    mem_used = format_bytes(memory.get('used_bytes'))
    mem_total = format_bytes(memory.get('total_bytes'))
    mem_pct = memory.get('usage_percent')
    mem_part = f'RAM {mem_used}/{mem_total}'
    if mem_pct is not None:
        mem_part += f' ({mem_pct:.1f}%)'
    gpu = runtime.get('gpu')
    if gpu and gpu.get('devices'):
        device = gpu['devices'][0]
        gpu_part = (
            f"GPU {device['name']} {device['memory_used_mb']}/{device['memory_total_mb']} MB "
            f"{device['utilization_gpu_percent']}%"
        )
    else:
        gpu_part = 'GPU none'
    return f'{mem_part} | {gpu_part}'


def read_notebook_progress(log_path: Path) -> str | None:
    if not log_path.exists():
        return None
    text = log_path.read_text(encoding='utf-8', errors='replace').replace('\r', '\n')
    for line in reversed(text.splitlines()):
        cleaned = line.strip()
        if cleaned.startswith('Executing:'):
            return cleaned
    return None


def write_heartbeat(state: str = 'idle', extra: dict | None = None) -> dict:
    payload = {
        'timestamp_utc': utc_now().isoformat(),
        'state': state,
        'poll_seconds': POLL_SECONDS,
        'heartbeat_interval_seconds': HEARTBEAT_INTERVAL_SECONDS,
        'control_path': str(CONTROL_PATH),
        'control_stop_requested': stop_requested(),
        'inbox_dir': str(INBOX_DIR),
        'running_jobs': len(list(RUNNING_DIR.glob('*'))),
        'pending_jobs': len(list_pending_jobs()),
        'runtime': get_runtime_stats(),
    }
    if extra:
        payload.update(extra)
    write_json(HEARTBEAT_PATH, payload)
    return payload


def execute_job(job_path: Path) -> dict:
    job_id, job_dir, work_root = build_job_paths(job_path)
    job_dir.mkdir(parents=True, exist_ok=False)
    shutil.move(str(job_path), str(work_root))

    log_path = job_dir / 'execution.log'
    started_at = utc_now()
    status = 'failed'
    return_code = None
    stop_reason = None
    entrypoint = None
    entrypoint_relative = None
    command = None
    executed_notebook = None
    executed_notebook_relative = None
    last_progress_log = 0.0
    last_progress_message = None

    with log_path.open('w', encoding='utf-8') as log_file:
        log_file.write(f'job_id: {job_id}\n')
        log_file.write(f'job_name: {job_path.name}\n')
        log_file.write(f'started_at_utc: {started_at.isoformat()}\n\n')
        log_file.flush()

        try:
            entrypoint = resolve_job_entrypoint(work_root)
            entrypoint_relative = entrypoint.relative_to(job_dir)
            command = build_job_command(entrypoint, job_dir)
            if entrypoint.suffix.lower() == '.ipynb':
                executed_notebook = entrypoint.with_name(f"{entrypoint.stem}.executed{entrypoint.suffix}")
                executed_notebook_relative = executed_notebook.relative_to(job_dir)
            log_file.write(f'entrypoint: {entrypoint_relative}\n')
            log_file.write(f'job_type: {entrypoint.suffix.lower()}\n')
            log_file.write(f'command: {command}\n\n')
            if stop_requested():
                stop_reason = 'control.md requested stop before launch'
                log_file.write(f'{stop_reason}\n')
                status = 'stopped'
            else:
                process = subprocess.Popen(
                    command,
                    cwd=str(job_dir),
                    stdout=log_file,
                    stderr=subprocess.STDOUT,
                    text=True,
                )
                deadline = time.time() + JOB_TIMEOUT_SECONDS
                while True:
                    return_code = process.poll()
                    if return_code is not None:
                        status = 'done' if return_code == 0 else 'failed'
                        break
                    if stop_requested():
                        stop_reason = 'control.md requested stop during execution'
                        log_file.write(f'\n{stop_reason}. Terminating process...\n')
                        process.terminate()
                        try:
                            return_code = process.wait(timeout=10)
                        except subprocess.TimeoutExpired:
                            log_file.write('Process did not exit after terminate(); killing it.\n')
                            process.kill()
                            return_code = process.wait()
                        status = 'stopped'
                        break
                    if time.time() >= deadline:
                        log_file.write(f'\nJob exceeded timeout of {JOB_TIMEOUT_SECONDS} seconds.\n')
                        process.terminate()
                        try:
                            return_code = process.wait(timeout=10)
                        except subprocess.TimeoutExpired:
                            log_file.write('Process did not exit after timeout terminate(); killing it.\n')
                            process.kill()
                            return_code = process.wait()
                        status = 'failed'
                        break
                    if entrypoint and entrypoint.suffix.lower() == '.ipynb':
                        now = time.time()
                        if now - last_progress_log >= NOTEBOOK_PROGRESS_INTERVAL_SECONDS:
                            progress = read_notebook_progress(log_path) or 'Executing notebook...'
                            progress_message = (
                                f'Notebook progress [{job_path.name}] {progress} | '
                                f"{format_runtime_brief(get_runtime_stats())}"
                            )
                            if progress_message != last_progress_message:
                                print(progress_message)
                                last_progress_message = progress_message
                            last_progress_log = now
                    time.sleep(1)
        except Exception:
            log_file.write('\nUnexpected runner error:\n')
            log_file.write(traceback.format_exc())
            status = 'failed'

        finished_at = utc_now()
        log_file.write(f'\nfinished_at_utc: {finished_at.isoformat()}\n')
        log_file.write(f'status: {status}\n')
        log_file.write(f'return_code: {return_code}\n')
        if stop_reason:
            log_file.write(f'stop_reason: {stop_reason}\n')

    if status == 'done':
        target_root = DONE_DIR
    elif status == 'stopped':
        target_root = FAILED_DIR
    else:
        target_root = FAILED_DIR
    final_dir = target_root / job_id
    shutil.move(str(job_dir), str(final_dir))

    metadata = {
        'job_id': job_id,
        'job_name': job_path.name,
        'job_type': entrypoint.suffix.lower() if entrypoint else None,
        'entrypoint': str(entrypoint_relative) if entrypoint_relative else None,
        'status': status,
        'return_code': return_code,
        'started_at_utc': started_at.isoformat(),
        'finished_at_utc': finished_at.isoformat(),
        'log_path': str(final_dir / 'execution.log'),
        'result_dir': str(final_dir),
        'stop_reason': stop_reason,
        'executed_notebook': str(executed_notebook_relative) if executed_notebook_relative else None,
    }
    write_json(final_dir / 'job_result.json', metadata)
    return metadata


def run_pending_jobs() -> list[dict]:
    results = []
    for job_path in list_pending_jobs():
        print(f'Running {job_path.name} ...')
        result = execute_job(job_path)
        print(f"Finished {result['job_name']} with status={result['status']}")
        results.append(result)
    if not results:
        print('No pending jobs found.')
    return results


## Optional: create a sample bundled job

Run the next cell once if you want a test job folder with a declared entrypoint to appear in `jobs/inbox/`.


In [ ]:
sample_job = INBOX_DIR / 'example_bundle'
if not sample_job.exists():
    sample_job.mkdir(parents=True, exist_ok=False)
    (sample_job / 'job.json').write_text(
        json.dumps({'entrypoint': 'main.py'}, indent=2) + '\n',
        encoding='utf-8',
    )
    (sample_job / 'main.py').write_text(
        """from helper import message\n"
        "from pathlib import Path\n\n"
        "print(message())\n"
        "artifact = Path('artifact.txt')\n"
        "artifact.write_text('Job completed successfully\\n', encoding='utf-8')\n"
        "print(f'Wrote {artifact.resolve()}')\n""",
        encoding='utf-8',
    )
    (sample_job / 'helper.py').write_text(
        "def message() -> str:\n"
        "    return 'Hello from bundled Colab Agent job'\n",
        encoding='utf-8',
    )
    print(f'Created sample bundled job at {sample_job}')
else:
    print(f'Sample job already exists at {sample_job}')


## Run once or watch continuously

Use `run_pending_jobs()` for a single pass, or `watch_forever()` to keep polling Drive while the Colab runtime stays active.


In [ ]:
def watch_forever(
    poll_seconds: int = POLL_SECONDS,
    heartbeat_interval_seconds: int = HEARTBEAT_INTERVAL_SECONDS,
) -> None:
    print(f'Starting watcher with poll interval = {poll_seconds} seconds')
    print(f'Heartbeat file: {HEARTBEAT_PATH}')
    print(f'Control file: {CONTROL_PATH}')
    print(f'Heartbeat interval = {heartbeat_interval_seconds} seconds')
    print(f'Drop .py files, .ipynb files, or job folders into {INBOX_DIR}')
    last_heartbeat = 0.0
    while True:
        now = time.time()
        if now - last_heartbeat >= heartbeat_interval_seconds:
            state = 'stop-requested' if stop_requested() else 'watching'
            heartbeat = write_heartbeat(state=state)
            print(
                f"Heartbeat at {heartbeat['timestamp_utc']} | "
                f"{format_runtime_brief(heartbeat.get('runtime'))}"
            )
            last_heartbeat = now
        if stop_requested():
            print(f'Stop requested in {CONTROL_PATH}; watcher is staying alive and will not launch new jobs.')
            time.sleep(poll_seconds)
            continue
        pending = list_pending_jobs()
        if pending:
            print(f'Found {len(pending)} pending job(s) at {utc_now().isoformat()}')
            results = run_pending_jobs()
            latest = results[-1] if results else None
            write_heartbeat(state='watching', extra={'last_result': latest})
        else:
            print(f"No jobs at {utc_now().isoformat()} | {format_runtime_brief(get_runtime_stats())}")
        time.sleep(poll_seconds)


# Single pass:
# write_heartbeat(state='single-pass-start')
# run_pending_jobs()

# Continuous watcher:
# watch_forever()
